# 运营仪表盘模板

这个 notebook 用于 business review / 运营复盘场景，复用现有的 notebook 框架中的聚合、时序、漏斗、分层和归因能力。

In [6]:
from pathlib import Path
import pandas as pd

from notebook_framework import (
    NotebookProject,
    build_report_table,
    crosstab_summary,
    funnel_analysis,
    plot_metric_comparison,
    time_series_summary,
    attribution_summary,
    user_segmentation,
)

repo_root = Path.cwd().resolve().parent.parent
project = NotebookProject(repo_root=repo_root)
df = project.load_csv('notebook_env_demo.csv')
df['date'] = pd.to_datetime(df['date'])
df.head()

,date,source,region,channel,visits,orders,revenue,conversion
0,2026-01-01,search,north,organic,836,71,3474.88,0.0849
1,2026-01-01,ads,north,paid,924,60,2233.92,0.0649
2,2026-01-01,social,north,social,731,32,890.48,0.0438
3,2026-01-01,email,north,email,676,74,4346.08,0.1095
4,2026-01-02,search,south,organic,786,66,3230.88,0.0840


In [7]:
summary = build_report_table(
    df,
    group_by='source',
    metrics={'visits': 'visits', 'orders': 'orders', 'revenue': 'revenue'},
)
summary

,source,visits,orders,revenue
0,ads,38934,2516,93690.72
1,search,34273,2899,141893.84
2,social,31396,1399,38885.68
3,email,29879,3272,192166.32


In [8]:
source_trend = time_series_summary(df, date_col='date', value_col='visits', group_by='source', freq='D')
source_trend.head()

,source,date,total_visits
0,ads,2026-01-01,924
1,ads,2026-01-02,876
2,ads,2026-01-03,1046
3,ads,2026-01-04,1263
4,ads,2026-01-05,1164


In [9]:
cross = crosstab_summary(
    df,
    index_col='source',
    columns_col='region',
    values_col='revenue',
    aggfunc='sum',
)
cross

region,east,north,south
source,,,
ads,31356.16,33550.24,28784.32
email,64486.40,68656.08,59023.84
search,47478.40,50853.76,43561.68
social,13036.40,13898.08,11951.20


In [10]:
funnel_df = df[['user_id', 'source', 'channel']].drop_duplicates().copy()
funnel_df['stage'] = 'view'
funnel = funnel_analysis(
    pd.DataFrame({'user_id': df['user_id'], 'stage': 'view'}),
    user_id_col='user_id',
    stage_col='stage',
)
funnel

KeyError: "['user_id'] not in index"

In [ ]:
segments = user_segmentation(
    df.groupby('user_id', as_index=False).agg(revenue=('revenue', 'sum'))
    ,
    user_id_col='user_id',
    value_col='revenue',
)
segments.head()

In [ ]:
attribution = attribution_summary(
    df,
    user_id_col='user_id',
    channel_col='channel',
    value_col='revenue',
    conversion_col='orders',
    method='last_touch',
)
attribution

In [ ]:
fig, ax = plot_metric_comparison(
    summary,
    x_col='source',
    metrics=['visits', 'orders'],
    title='渠道流量与订单对比',
)
fig